# 🚀 AlloyDB ScaNN Vector Search Benchmark (Auto Mode)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GoogleCloudPlatform/python-docs-samples/blob/main/alloydb/notebooks/alloydb_scann_benchmark_glove.ipynb)

---
This interactive notebook benchmarks the vector search performance (**Recall vs QPS**) of **AlloyDB ScaNN** in **Auto Mode** across percentage-based partitions using **`scann.pct_leaves_to_search`** on the **GloVe-100 Angular** dataset (1.18M vectors, 100 dimensions).

📚 **Helpful Resources:**
* [Create and Manage ScaNN Indexes in Auto Mode](https://cloud.google.com/alloydb/docs/ai/create-scann-index#create-scann-index-automatic)
* [Perform Vector Search in AlloyDB](https://cloud.google.com/alloydb/docs/ai/perform-vector-search)
* [Tune Vector Indexes](https://cloud.google.com/alloydb/docs/ai/tune-indexes)
* [Measure Vector Query Recall](https://cloud.google.com/alloydb/docs/ai/measure-vector-query-recall)
---

In [ ]:
# @title ⚙️ Benchmark Execution
# ==========================================
# 1. CONFIGURATION
# ==========================================
# @markdown ### **⚙️ 1. Cluster Configuration**
project_id = ""  # @param {type:"string", placeholder:"Project ID"}
region = ""  # @param {type:"string", placeholder:"Region (e.g. us-central1)"}
cluster_id = ""  # @param {type:"string", placeholder:"Cluster name"}
instance_id = ""  # @param {type:"string", placeholder:"Instance Name"}

# @markdown ### **🔐 2. Database Credentials**
# @markdown *Note: Password will be requested securely when the benchmark is executed.*
db_user = "postgres"  # @param {type:"string", placeholder:"DB User"}
db_name = "postgres"  # @param {type:"string", placeholder:"DB Name"}

# @markdown ### **⚡ 3. Benchmark Parameters**
# @markdown * `num_queries`: Total test queries to run.
# @markdown * `top_k`: Number of nearest neighbors to retrieve (LIMIT).
num_queries = 100  # @param [100, 500, 1000, 10000] {type:"raw"}
top_k = 10  # @param [10, 20, 50, 100] {type:"raw"}

# ==========================================
# 2. SETUP & AUTHENTICATION
# ==========================================
import getpass
import os
import shutil
import sys
import time
import urllib.request

# Prompt for password upfront so user doesn't wait for pip installation
db_pass = getpass.getpass("🔑 Enter Database Password for AlloyDB: ")

# Install dependencies silently (standard for Colab VM runtimes)
try:
  import pg8000
  import google.cloud.alloydb.connector
  import h5py
  import matplotlib
  import tqdm
except ImportError:
  try:
    # Run pip install in standard Colab environment
    get_ipython().system(
        "pip install -q pg8000 google-cloud-alloydb-connector[pg8000] tqdm"
        " matplotlib h5py numpy"
    )
  except Exception as e:
    print(f"⚠️ Could not auto-install packages: {e}")
    print(
        "ℹ️ Ensure you are connected to a standard Colab runtime (e.g. on"
        " colab.sandbox.google.com or colab.research.google.com)."
    )

# Authenticate Colab session
try:
  from google.colab import auth

  auth.authenticate_user()
except Exception as e:
  print(f"ℹ️ Google Cloud auth note: {e}")

from IPython.display import clear_output, display, HTML

clear_output()

# ==========================================
# 3. LATE IMPORTS (Post-Installation) & THEME-ADAPTIVE STYLING
# ==========================================
import h5py
import matplotlib.patheffects as patheffects
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from tqdm.notebook import tqdm
from google.cloud.alloydb.connector import Connector


def print_step(step_label, text="", is_success=False, is_error=False):
  """Renders status messages with styling tuned for both light and dark backgrounds."""
  base_style = "font-size: 16px; margin-top: 10px; margin-bottom: 4px;"
  if is_success:
    display(
        HTML(f"""
        <div style="{base_style} font-weight: bold; color: #34A853;">
            {step_label} {text}
        </div>
        """)
    )
  elif is_error:
    display(
        HTML(f"""
        <div style="{base_style} font-weight: bold; color: #EA4335;">
            {step_label} {text}
        </div>
        """)
    )
  else:
    display(
        HTML(f"""
        <div style="{base_style}">
            <span style="color: var(--colab-anchor-color, #1a73e8); font-weight: bold;">{step_label}</span>
            <span style="color: var(--colab-primary-text-color, #202124);">{text}</span>
        </div>
        """)
    )


# ==========================================
# 4. DATASET CONFIGURATION (GloVe-100 Angular)
# ==========================================
DATASET_CONFIG = {
    "url": "https://ann-benchmarks.com/glove-100-angular.hdf5",
    "file_name": "glove-100-angular.hdf5",
    "dim": 100,
    "table_name": "glove",
    "test_table_name": "glove_test",
    "distance_op": "<=>",
    "index_metric": "cosine",
    "description": "GloVe-100 Angular (1.18M train, 10k test, dim=100)",
}


# ==========================================
# 5. CORE HELPER FUNCTIONS
# ==========================================
def download_dataset(dataset_cfg):
  file_name = dataset_cfg["file_name"]
  if not os.path.exists(file_name):
    url = dataset_cfg["url"]
    print_step(
        "📥",
        f"Downloading {dataset_cfg['description']} from {url}...",
    )
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with (
        urllib.request.urlopen(req, timeout=60) as response,
        open(file_name, "wb") as out_file,
    ):
      shutil.copyfileobj(response, out_file)
    print_step("✅", f"Dataset {file_name} downloaded successfully.", is_success=True)
  else:
    print_step(
        "ℹ️",
        f"Dataset {file_name} already exists locally. Skipping download.",
    )


def setup_schema(cur, dataset_cfg):
  dim = dataset_cfg["dim"]
  tbl = dataset_cfg["table_name"]
  tbl_test = dataset_cfg["test_table_name"]

  cur.execute("CREATE EXTENSION IF NOT EXISTS vector CASCADE;")
  cur.execute("CREATE EXTENSION IF NOT EXISTS alloydb_scann CASCADE;")
  cur.execute(f"DROP TABLE IF EXISTS {tbl} CASCADE;")
  cur.execute(f"DROP TABLE IF EXISTS {tbl_test} CASCADE;")
  cur.execute(
      f"CREATE TABLE {tbl} (id bigserial PRIMARY KEY, embedding vector({dim}));"
  )
  cur.execute(
      f"CREATE TABLE {tbl_test} (id bigserial PRIMARY KEY, embedding"
      f" vector({dim}));"
  )


def stream_bulk_insert(
    cur,
    table_name,
    column_name,
    numpy_matrix,
    batch_size=5000,
    desc="Inserting",
):
  total_rows = len(numpy_matrix)
  for i in tqdm(
      range(0, total_rows, batch_size),
      desc=desc,
      leave=True,
      colour="#1f77b4",
  ):
    batch = numpy_matrix[i : i + batch_size]
    placeholders = ", ".join(["(%s)"] * len(batch))
    params = [str(vec.tolist()) for vec in batch]
    cur.execute(
        f"INSERT INTO {table_name} ({column_name}) VALUES {placeholders}",
        params,
    )


def ingest_dataset(cur, dataset_cfg):
  file_name = dataset_cfg["file_name"]
  tbl = dataset_cfg["table_name"]
  tbl_test = dataset_cfg["test_table_name"]

  with h5py.File(file_name, "r") as f:
    train_data = f["train"][:]
    test_data = f["test"][:]

  stream_bulk_insert(
      cur,
      tbl,
      "embedding",
      train_data,
      batch_size=5000,
      desc=f"Inserting {len(train_data):,} Training Records",
  )
  stream_bulk_insert(
      cur,
      tbl_test,
      "embedding",
      test_data,
      batch_size=5000,
      desc=f"Inserting {len(test_data):,} Test Queries",
  )


def build_scann_index(cur, dataset_cfg, index_name="my_scann_idx"):
  tbl = dataset_cfg["table_name"]
  metric = dataset_cfg["index_metric"]

  # VACUUM ANALYZE to ensure accurate statistics for ScaNN AUTO mode
  cur.execute(f"VACUUM (DISABLE_PAGE_SKIPPING, ANALYZE) {tbl};")
  cur.execute(
      f"VACUUM (DISABLE_PAGE_SKIPPING, ANALYZE) {dataset_cfg['test_table_name']};"
  )

  cur.execute(f"DROP INDEX IF EXISTS {index_name};")

  # Dynamically configure parallel workers based on instance max_worker_processes
  cur.execute("SELECT current_setting('max_worker_processes')::int;")
  max_workers = cur.fetchone()[0]
  cur.execute(f"SET max_parallel_workers = {max_workers};")
  cur.execute(f"SET max_parallel_maintenance_workers = {max_workers};")

  # Scale maintenance_work_mem to at least 10% of table size (minimum 2GB for fast in-memory sampling)
  cur.execute(f"SELECT pg_relation_size('{tbl}');")
  table_size_bytes = cur.fetchone()[0]
  maint_mem_mb = max(2048, int((table_size_bytes * 0.10) / (1024 * 1024)))
  cur.execute(f"SET maintenance_work_mem = '{maint_mem_mb}MB';")

  t0 = time.time()
  cur.execute(
      f"CREATE INDEX {index_name} ON {tbl} USING scann (embedding {metric}) WITH"
      " (mode = 'AUTO');"
  )
  build_duration_sec = time.time() - t0

  cur.execute(f"SELECT pg_size_pretty(pg_relation_size('{index_name}'));")
  index_size = cur.fetchone()[0]
  cur.execute(f"SELECT pg_size_pretty(pg_relation_size('{tbl}'));")
  table_size = cur.fetchone()[0]

  print_step(
      "✅",
      f"ScaNN index '{index_name}' built in {build_duration_sec:.2f}s | "
      f"Index Size: {index_size} (Table Size: {table_size})",
      is_success=True,
  )


def deploy_benchmark_functions(cur, dataset_cfg):
  tbl = dataset_cfg["table_name"]
  tbl_test = dataset_cfg["test_table_name"]
  op = dataset_cfg["distance_op"]

  # 1. Recall measurement function (using evaluate_query_recall and scann.pct_leaves_to_search)
  cur.execute(f"""
    CREATE OR REPLACE FUNCTION measure_recall(pct_leaves_to_search FLOAT, num_q INT, k INT)
    RETURNS FLOAT AS $func$
    DECLARE
        q_vec text;
        query_str text;
        total_recall FLOAT := 0.0;
        valid_queries INT := 0;
        v_id int; v_q text; v_p json; v_recall float; v_at float; v_et float; v_idx text;
    BEGIN
        IF num_q <= 0 THEN RETURN 0.0; END IF;

        FOR q_vec IN SELECT embedding::text FROM {tbl_test} LIMIT num_q LOOP
            query_str := pg_catalog.format('SELECT id FROM {tbl} ORDER BY embedding {op} ''%s'' LIMIT %s', q_vec, k);
            BEGIN
                EXECUTE pg_catalog.format('SELECT * FROM evaluate_query_recall($$%s$$, ''{{"scann.pct_leaves_to_search": %s, "scann.num_leaves_to_search": 0}}'', ''{{"scann"}}'')', query_str, pct_leaves_to_search)
                    INTO v_id, v_q, v_p, v_recall, v_at, v_et, v_idx;
                IF v_recall IS NOT NULL THEN
                    total_recall := total_recall + v_recall;
                    valid_queries := valid_queries + 1;
                END IF;
            EXCEPTION
                WHEN OTHERS THEN
                    RAISE WARNING 'Error during evaluate_query_recall: %', SQLERRM;
            END;
        END LOOP;

        IF valid_queries = 0 THEN RETURN 0.0; END IF;
        RETURN total_recall / valid_queries;
    END;
    $func$ LANGUAGE plpgsql;
    """)

  # 2. QPS measurement function (using scann.pct_leaves_to_search)
  cur.execute(f"""
    CREATE OR REPLACE FUNCTION measure_qps(pct_leaves_to_search FLOAT, num_q INT, k INT)
    RETURNS FLOAT AS $func$
    DECLARE
        q_vec vector;
        start_time timestamp;
        end_time timestamp;
        total_time_ms float;
    BEGIN
        IF num_q <= 0 THEN RETURN 0.0; END IF;

        PERFORM pg_catalog.set_config('scann.num_leaves_to_search', '0', false);
        PERFORM pg_catalog.set_config('scann.pct_leaves_to_search', pct_leaves_to_search::text, false);

        start_time := pg_catalog.clock_timestamp();
        FOR q_vec IN SELECT embedding FROM {tbl_test} LIMIT num_q LOOP
            EXECUTE pg_catalog.format('SELECT id FROM {tbl} ORDER BY embedding {op} $1 LIMIT %s', k) USING q_vec;
        END LOOP;
        end_time := pg_catalog.clock_timestamp();

        total_time_ms := (EXTRACT(epoch FROM end_time) - EXTRACT(epoch FROM start_time)) * 1000.0;
        IF total_time_ms <= 0.0 THEN RETURN 0.0; END IF;

        RETURN (num_q * 1000.0) / total_time_ms;
    END;
    $func$ LANGUAGE plpgsql;
    """)


def run_benchmark_sweep(cur, pct_values, num_queries, top_k):
  print_step("🔥", "Warming up database cache (0.5% leaves to search)...")
  # Configure query execution memory and parallelism once for the benchmark session
  cur.execute("SET work_mem = '256MB';")
  cur.execute("SET max_parallel_workers_per_gather = 4;")

  cur.execute(
      "SELECT measure_qps(0.5, %s, %s);",
      (min(50, num_queries), top_k),
  )

  results = []
  for pct in tqdm(
      pct_values,
      desc="Benchmarking ScaNN Auto Search (% Leaves)",
      colour="#1f77b4",
  ):
    cur.execute(
        "SELECT measure_recall(%s, %s, %s);",
        (pct, num_queries, top_k),
    )
    recall = cur.fetchone()[0]
    cur.execute(
        "SELECT measure_qps(%s, %s, %s);",
        (pct, num_queries, top_k),
    )
    qps = cur.fetchone()[0]
    results.append({"pct": pct, "recall": recall, "qps": qps})
  return results


def display_summary_table(results, top_k):
  print_step("📊", "Benchmark Summary Table:", is_success=True)
  rows_html = "".join([
      f"""<tr style="border-bottom: 1px solid #e0e0e0;">
            <td style="padding: 8px 16px; text-align: right;">{r['pct']}%</td>
            <td style="padding: 8px 16px; text-align: right; color: {'#34A853' if r['recall'] >= 0.9 else 'inherit'}; font-weight: {'bold' if r['recall'] >= 0.9 else 'normal'};">{r['recall']:.4f}</td>
            <td style="padding: 8px 16px; text-align: right; font-weight: bold;">{r['qps']:.1f}</td>
        </tr>"""
      for r in results
  ])
  table_html = f"""
    <table style="border-collapse: collapse; width: 85%; max-width: 650px; margin: 12px 0; font-size: 14px; font-family: monospace, sans-serif; color: var(--colab-primary-text-color, #202124);">
        <thead>
            <tr style="border-bottom: 2px solid #cccccc; background-color: rgba(66, 133, 244, 0.08);">
                <th style="padding: 10px 16px; text-align: right;">pct_leaves_to_search</th>
                <th style="padding: 10px 16px; text-align: right;">Recall @ {top_k}</th>
                <th style="padding: 10px 16px; text-align: right;">QPS (Queries/sec)</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    """
  display(HTML(table_html))


def plot_results(results, dataset_cfg, top_k):
  recalls = [r["recall"] for r in results]
  qps_vals = [r["qps"] for r in results]
  pct_labels = [r["pct"] for r in results]

  plt.rc("font", size=14)
  plt.rc("axes", titlesize=16)
  plt.rc("axes", labelsize=14)
  plt.rc("xtick", labelsize=12)
  plt.rc("ytick", labelsize=12)
  plt.rc("legend", fontsize=13)

  fig, ax = plt.subplots(figsize=(12, 7), dpi=120)

  color_scann = "#1f77b4"

  ax.plot(
      recalls,
      qps_vals,
      marker="o",
      linewidth=2.5,
      markersize=9,
      color=color_scann,
      label="AlloyDB ScaNN (Auto Mode)",
  )
  ax.fill_between(recalls, qps_vals, color=color_scann, alpha=0.08)

  ax.set_xlabel(f"Recall @ {top_k}", fontweight="bold", labelpad=10)
  ax.set_ylabel("Queries Per Second (QPS)", fontweight="bold", labelpad=10)
  ax.set_title(
      f"AlloyDB ScaNN: Recall vs QPS\n({dataset_cfg['description']}, mode=AUTO,"
      f" K={top_k})",
      fontsize=15,
      fontweight="bold",
      pad=20,
  )

  max_qps = max(qps_vals) if qps_vals else 100
  min_recall = min(recalls) if recalls else 0.5
  ax.set_ylim(0, max_qps * 1.18)
  ax.set_xlim(max(0.0, min_recall - 0.05), 1.02)

  ax.yaxis.set_major_locator(ticker.MaxNLocator(8))
  ax.xaxis.set_major_locator(ticker.MultipleLocator(0.05))

  # Clean spines for compatibility across light and dark themes
  ax.spines["top"].set_visible(False)
  ax.spines["right"].set_visible(False)
  ax.spines["left"].set_color("#cccccc")
  ax.spines["bottom"].set_color("#cccccc")

  plt.grid(True, which="both", color="#f0f0f0", linestyle="-", linewidth=1.5)

  pe = [patheffects.withStroke(linewidth=3, foreground="white", alpha=0.9)]

  for i in range(len(recalls)):
    ax.text(
        recalls[i],
        qps_vals[i] + (max_qps * 0.03),
        f"PCT={pct_labels[i]}%\n({qps_vals[i]:.0f} QPS)",
        color="#D97706",
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="bottom",
        path_effects=pe,
    )

  ax.legend(loc="lower left", frameon=True, edgecolor="#cccccc")
  fig.tight_layout()
  plt.show()


# ==========================================
# 6. MAIN EXECUTION PIPELINE
# ==========================================
connector = None
conn = None
cur = None

try:
  print_step("✅", "GCP Authentication & Setup Successful!", is_success=True)
  print_step(
      "🚀",
      "Starting AlloyDB ScaNN Benchmark Pipeline (Auto Mode - % Leaves"
      " Search)...",
  )

  dataset_cfg = DATASET_CONFIG

  # Step 1: Connect to AlloyDB
  print_step(
      "Step 1/7:",
      "🔌 Initializing Secure AlloyDB Connector...",
  )
  connector = Connector()
  conn = connector.connect(
      f"projects/{project_id}/locations/{region}/clusters/{cluster_id}/instances/{instance_id}",
      "pg8000",
      user=db_user,
      password=db_pass,
      db=db_name,
      ip_type="PUBLIC",
  )
  conn.autocommit = True
  cur = conn.cursor()

  # Step 2: Database Schema & Extensions
  print_step(
      "Step 2/7:",
      "🛠️ Setting up pgvector & alloydb_scann extensions...",
  )
  setup_schema(cur, dataset_cfg)

  # Step 3: Download Dataset
  print_step(
      "Step 3/7:",
      f"📥 Checking & Downloading {dataset_cfg['description']}...",
  )
  download_dataset(dataset_cfg)

  # Step 4: Stream Ingestion
  print_step(
      "Step 4/7:",
      f"📂 Ingesting training data & test queries into AlloyDB ({dataset_cfg['table_name']})...",
  )
  ingest_dataset(cur, dataset_cfg)

  # Step 5: Build ScaNN Index in AUTO Mode
  print_step(
      "Step 5/7:",
      "🏗️ Building ScaNN Index with mode='AUTO' (VACUUM ANALYZE &"
      " auto-tuning)...",
  )
  build_scann_index(cur, dataset_cfg)

  # Step 6: Deploy Benchmark Functions
  print_step(
      "Step 6/7:",
      "⚙️ Deploying Benchmark Functions (measure_recall & measure_qps)...",
  )
  deploy_benchmark_functions(cur, dataset_cfg)

  # Step 7: Sweep & Evaluation
  print_step(
      "Step 7/7:",
      "⏱️ Benchmarking ScaNN recall & QPS across search partition"
      " percentages...",
  )
  pct_values = [0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
  benchmark_results = run_benchmark_sweep(cur, pct_values, num_queries, top_k)

  # Display Table & Plot
  display_summary_table(benchmark_results, top_k)
  plot_results(benchmark_results, dataset_cfg, top_k)

except Exception as e:
  print_step("❌", f"ERROR: {str(e)}", is_error=True)

finally:
  if "cur" in locals() and cur:
    cur.close()
  if "conn" in locals() and conn:
    conn.close()
  if "connector" in locals() and connector:
    connector.close()
